## Analyzing single-cell RNA-seq data (clustering and cell type annotations)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Installation & Importing Packages

In [ ]:
!pip install scanpy #--target '/content/drive/MyDrive/Colab Notebooks/Colab_packages'

In [ ]:
#need to install these too
#!conda install -y -c anaconda cmake
!pip install leidenalg #--target '/content/drive/MyDrive/Colab Notebooks/Colab_packages'
!pip install louvain #--target '/content/drive/MyDrive/Colab Notebooks/Colab_packages'

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import scanpy as sc
import anndata

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0),                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

## Data Load

In [ ]:
adata = sc.read("/content/drive/MyDrive/Colab Notebooks/ESCA/SCG_2023/Week3/Melanoma_2000.h5ad")

In [ ]:
adata

In [ ]:
adata.obs['tumor'] = adata.obs['tumor'].astype('object')
adata.obs['Malignant'] = adata.obs['Malignant'].astype('object')
adata.obs['Non_malignant'] = adata.obs['Non_malignant'].astype('object')

In [ ]:
adata.obs.Malignant.value_counts()

In [ ]:
#remove unknowns
adata2 = adata[~(adata.obs["Malignant"]==0)]
adata2

## Data Filteration

In [ ]:
sc.pp.filter_cells(adata, min_genes=200) #get rid of cells with fewer than 200 genes
sc.pp.filter_genes(adata, min_cells=3) #get rid of genes that are found in fewer than 3 cells

In [ ]:
adata

In [ ]:
#IF YOU ARE DOING MOUSE YOU MIGHT NEED TO CHANGE MT- to Mt. Always double check you actually labeled MT
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'

In [ ]:
adata.var[adata.var.mt==True]

In [ ]:
adata.obs

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)


In [ ]:
adata.obs[adata.obs.index == "CY58_1_CD45_B02_S974_comb"]

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)


In [ ]:
#sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt') #No mit Percentage
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
#instead of picking subjectively, you can use quanitle
upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
lower_lim = np.quantile(adata.obs.n_genes_by_counts.values, .02)
print(f'{lower_lim} to {upper_lim}')

In [ ]:
#adata = adata[adata.obs.n_genes_by_counts < 7000, :] #example if you wanted to pick a number yourself
adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & (adata.obs.n_genes_by_counts > lower_lim)]
#adata = adata[adata.obs.pct_counts_mt < 20] #for mct  percentage

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)


In [ ]:
adata

In [ ]:
#Don't run for this data
#sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10,000 UMI

In [ ]:
#sc.pp.log1p(adata) #change to log counts

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )
#Show those genes that yield the highest fraction of counts in each single cell, across all cells.

### HVG

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5) #these are default values
#minimum mean expression threshold
#max mean expression threshold
#minimum dispersion threshold.

In [ ]:
adata.var[adata.var.highly_variable==True]

In [ ]:
adata.raw = adata #save raw data before processing values and further filtering

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata = adata[:, adata.var.highly_variable] #filter highly variable

In [ ]:
# The session crashed
#sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt']) #Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed

In [ ]:
sc.pp.scale(adata, max_value=10) #scale each gene to unit variance

In [ ]:
adata.X.min(), adata.raw.X.min()

## Dimensionality Reduction by PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata)

In [ ]:
# scatter plot in the PCA coordinates
sc.pl.pca_scatter(adata, color="total_counts")

##Computing the neighborhood graph


In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=8)


### Non_linear Dimensionality Reduction by UMAP


In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata)

In [ ]:
sc.tl.leiden(adata, resolution = 0.40)
sc.pl.umap(adata, color=['leiden'])

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden"],
    legend_loc="on data",
)

## Find Markers

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)

In [ ]:
sc.pl.umap(adata, color = ['leiden'], frameon = True, legend_loc = "on data")


In [ ]:
sc.pl.umap(adata, color = ['leiden','Malignant'], frameon = True, legend_loc = "on data")

In [ ]:
Markers = sc.get.rank_genes_groups_df(adata, None)
Markers = Markers[(Markers.pvals_adj < 0.05) & (Markers.logfoldchanges > .5)]
Markers

## Manual Cell Type Annotation

In [ ]:
# for i in range(0,11):
#   print(f'"{i}": "",')

In [ ]:
#Markers of Macrophage
sc.pl.umap(adata, color = ['CD163','CD14','CSF1R'], frameon = True, vmax = 5)
sc.pl.violin(adata, ['CD163','CD14','CSF1R'], groupby='leiden', use_raw = True)

#cluster 4 is Macrophage

In [ ]:
#EndoThelial Cells
sc.pl.umap(adata, color = ['PECAM1','VWF','CDH5'], frameon = True, vmax = 5)
sc.pl.violin(adata, ['PECAM1','VWF','CDH5'], groupby='leiden', use_raw = True)

# 7 is endothelial

In [ ]:
#CAFs
sc.pl.umap(adata, color = ['FAP','DCN','COL1A1','COL6A3'], frameon = True, vmax = 5)
sc.pl.violin(adata, ['FAP','DCN','COL1A1','COL6A3'], groupby='leiden', use_raw = True)

#Cluster 8 is CAFs


In [ ]:
#Melanoma
sc.pl.umap(adata, color = ['MIA','TYR','SLC45A2','CDH19','SLC24A5'], frameon = True, vmax = 5)
sc.pl.violin(adata, ['MIA','TYR','SLC45A2','CDH19','SLC24A5'], groupby='leiden', use_raw = True)

#Cluster 2,3,9,10 are melanomas

In [ ]:
#T-Cells Markers
sc.pl.umap(adata, color = ['CD2','CD3D','CD3E','CD3G'], frameon = True, vmax = 5)
#clusters 0,1,5,6

In [ ]:
#Check scores for CD8A
Markers[Markers.names == 'CD8A']

In [ ]:
#Check scores for CD4
Markers[Markers.names == 'CD4']

In [ ]:
#NK Cells
sc.pl.umap(adata, color = ["NKG7","GNLY"	,"GZMA","GZMB"], frameon = True, vmax = 5)
sc.pl.violin(adata, [ "NKG7","GNLY"	,"GZMA","GZMB"], groupby='leiden', use_raw = True)

#Cluster 0,1,5,6 has NK cells

In [ ]:
sc.pl.umap(adata, color = ['Malignant'], frameon = True, legend_loc = "on data")


In [ ]:
#CD8+ T Cells
sc.pl.umap(adata, color = ["CD8A","GZMB","CD2","CD27","CD5","CD69","CD28"], frameon = True, vmax = 5)
sc.pl.violin(adata, [ "CD8A","GZMB","CD2","CD27","CD5","CD69","CD28"], groupby='leiden', use_raw = True)

In [ ]:
Markers = Markers[(Markers.pvals_adj < 0.05) & (Markers.logfoldchanges > .5)]
cluster_1_markers = Markers.names[Markers.group == '1'][0:10].to_list()
cluster_1_markers

In [ ]:
#cluster_1_markers
sc.pl.umap(adata, color = cluster_1_markers, frameon = True, vmax = 5)
sc.pl.violin(adata, cluster_1_markers, groupby='leiden', use_raw = True)

In [ ]:
cell_type ={"0": "T_cells/NK",
"1": "T_cells/NK",
"2": "Mel",
"3": "Mel",
"4": "Macro", #Macrophage
"5": "T_cells/NK",
"6": "T_cells/NK",
"7": "Endo",  #Endothelial
"8": "CAFs",
"9": "Mel",
"10": "Mel"}

In [ ]:
adata.obs['cell_type'] = adata.obs.leiden.map(cell_type)


In [ ]:
sc.pl.umap(adata, color = ['cell_type'], frameon = True, legend_loc = "on data")


In [ ]:
sc.pl.umap(adata, color = ['cell_type','Non_malignant','leiden'], frameon = True, legend_loc = "on data")


## Saving the object as h5ad to Disk



In [ ]:
adata.obs['tumor'] = adata.obs['tumor'].astype('str')
adata.obs['Malignant'] = adata.obs['Malignant'].astype('str')
adata.obs['Non_malignant'] = adata.obs['Non_malignant'].astype('str')

In [ ]:
adata.write('adata.h5ad', compression="gzip")